# Lesson 04 - Työkalujen käyttö -suunnittelumalli

Tässä oppitunnissa opit **Työkalujen käyttö** -suunnittelumallin tekoälyagentteja varten Microsoft Agent Frameworkin (Python) avulla. Käymme läpi:

- Funktiotyökalujen määrittelyn `@tool`-koristelijalla ja tyypitetyillä parametreilla  
- Työkalujen skeemojen tarjoamisen, jotta malli tietää mitä kukin työkalu tekee  
- Työkalujen suorittamisen hallinnan `approval_mode`-asetuksella  
- **Rakenteellisen tulosteen** palauttamisen Pydantic-mallien ja `response_format`-asetuksen avulla  

Skenaario on **matkavarauksen agentti**, joka voi etsiä kohteita, tarkistaa saatavuuden ja hakea lentotietoja.


## Asennus


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Työkalujen määrittely @tool -koristeella

`@tool`-koriste muuttaa tavallisen Python-funktion työkaluksi, jota agentti voi kutsua.
Tärkeimmät kohdat:

- **Docstring** muuttuu työkalun kuvaukseksi, jonka malli näkee.
- **Tyyppimerkinnät** (mukaan lukien `Annotated` kuvauksineen) määrittelevät työkalun skeeman.
- `approval_mode` ohjaa, pitääkö käyttäjän hyväksyä jokainen kutsu ennen sen suorittamista.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Agentin luominen useilla työkaluilla

Anna kaikki kolme työkalua asiakkaalle, jotta malli voi kutsua tarvitsemansa vastatakseen käyttäjän kysymykseen.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Jäsennelty tuloste työkaluilla

Asettamalla `response_format` Pydantic-malliksi agentti pakotetaan palauttamaan hyvin tyypitetty JSON-objekti vapaamuotoisen tekstin sijaan. Tämä on hyödyllistä, kun alempi koodi tarvitsee käsitellä tulosta ohjelmallisesti.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Työkalun hyväksymismallit

`approval_mode`-parametri `@tool`-merkinnässä ohjaa, vaaditaanko työkalukutsuilta ihmisen hyväksyntä ennen suorittamista:

| Tila | Käytös |
|---|---|
| `"never_require"` | Työkalu toimii automaattisesti — käyttäjän vahvistusta ei tarvita. |
| `"always_require"` | Jokainen kutsu on hyväksyttävä käyttäjän toimesta ennen suorittamista. |

Käytä `"always_require"`-asetusta työkaluissa, joilla on sivuvaikutuksia (esim. lennon varaaminen, luottokortin veloitus), jotta ihminen pysyy mukana prosessissa.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Yhteenveto

Tässä oppitunnissa opit miten:

1. **Määritellä työkaluja** käyttämällä `@tool`-koristetta tyypitetyillä parametreilla ja docstringeillä, jotka toimivat työkalun skeemana.
2. **Yhdistää useita työkaluja** siten, että agentti voi kutsua niitä peräkkäin vastatakseen monimutkaisiin kyselyihin.
3. **Palauttaa jäsenneltyä tulosta** välittämällä Pydantic-mallin `response_format`-parametrina.
4. **Hallita työkalun hyväksyntää** `approval_mode`-asetuksella, jotta ihminen pysyy mukana herkkien toimintojen valvonnassa.

Nämä mallit muodostavat perustan luotettavien, tuotantovalmiiden agenttien rakentamiseen, jotka voivat turvallisesti olla vuorovaikutuksessa ulkoisten järjestelmien kanssa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:  
Tämä asiakirja on käännetty tekoälyllä käyttäen käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Pyrimme tarkkuuteen, mutta ole hyvä ja huomioi, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäisellä kielellä on katsottava viralliseksi lähteeksi. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä johtuvista väärinymmärryksistä tai virhetulkintojen seurauksista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
